# Donut Base Model Test

`naver-clova-ix/donut-base` **(fine-tuning 되지 않은 사전학습 모델)** 로 CORD-v2 영수증 이미지를 추론합니다.

이 노트북의 목적은 **base 모델이 특정 태스크(영수증 파싱)에 대해 학습되지 않았을 때 어떻게 동작하는지** 직접 확인하는 것입니다.

> 비교 대상: [`donut_CORD_v2_fine_tunned_test.ipynb`](./donut_CORD_v2_fine_tunned_test.ipynb) — 같은 이미지를 **fine-tuned 모델**로 추론합니다.

| 구분 | Base Model (이 노트북) | Fine-tuned Model |
|------|------------------------|------------------|
| 모델 | `donut-base` | `donut-base-finetuned-cord-v2` |
| 학습 | IIT-CDIP 문서로 **텍스트 읽기**만 사전학습 | CORD-v2 영수증으로 **JSON 파싱** 추가 학습 |
| 태스크 토큰 | 없음 | `<s_cord-v2>` |
| 예상 출력 | 구조화되지 않은 텍스트 (파싱 실패) | `gt_parse` 형태의 구조화 JSON |

In [ ]:
import sys, torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()

from datasets import load_dataset
from PIL import Image
import torch, json, re
import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
# ── matplotlib 한글 폰트 설정 ──────────────────────────────────────
# 기본 폰트(DejaVu Sans)에는 한글 글리프가 없어 그래프 제목의 한글이 깨집니다.
# 시스템에 설치된 Noto Sans CJK 폰트를 등록해 한글을 정상 표시합니다.
import matplotlib.font_manager as fm

_font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
try:
    fm.fontManager.addfont(_font_path)
    # .ttc의 첫 face 이름('Noto Sans CJK JP')에도 한글 글리프가 포함돼 있습니다
    _font_name = fm.FontProperties(fname=_font_path).get_name()
    plt.rcParams["font.family"] = _font_name
    print(f"한글 폰트 적용: {_font_name}")
except FileNotFoundError:
    print(f"폰트 없음: {_font_path} — 제목 한글이 깨질 수 있습니다")

plt.rcParams["axes.unicode_minus"] = False  # 음수 기호(-) 깨짐 방지

## Base Model 이란

`naver-clova-ix/donut-base` 는 대량의 문서 이미지로 **일반적인 읽기 능력**만 사전학습한 모델입니다.

모델 페이지: https://huggingface.co/naver-clova-ix/donut-base

| 항목 | 내용 |
|------|------|
| 학습 데이터 | IIT-CDIP (1100만 장의 다양한 문서 이미지) |
| 학습 목표 | 이미지 속 텍스트를 **순서대로 읽는** 능력 (pseudo-OCR) |
| 결과 | 문서를 "보고 읽을" 수 있지만, **영수증 → JSON** 같은 특정 태스크는 모름 |

### 핵심 차이

fine-tuned 모델은 `<s_cord-v2>` 라는 **태스크 토큰**을 디코더 시작 토큰으로 받아 "이제 영수증을 파싱하라"는 신호를 인식합니다.

base 모델은 이 토큰 자체를 학습한 적이 없습니다. 따라서:
- `<s_cord-v2>` 토큰이 토크나이저 사전(vocab)에 **존재하지 않으며**,
- 영수증을 줘도 `gt_parse` 같은 구조화된 JSON을 만들지 못합니다.

```
Base Model   = 대학 졸업생  (글은 읽지만 영수증 처리 업무는 모름)
Fine-tuning  = 입사 후 실무 교육  (영수증 처리만 집중 훈련)
Fine-tuned   = 영수증 처리 전문 직원  (영수증 → JSON 즉시 변환)
```

In [ ]:
model_name = "naver-clova-ix/donut-base"

# DonutProcessor: 이미지 전처리(resize, normalize) + 토크나이저를 하나로 묶은 클래스
processor = DonutProcessor.from_pretrained(model_name)

# VisionEncoderDecoderModel: Swin Transformer(encoder) + mBART(decoder) 구조
# fine-tuned 모델과 구조는 동일하지만, 디코더 가중치가 태스크 학습 전 상태입니다.
model = VisionEncoderDecoderModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()  # 추론 모드: dropout 비활성화, gradient 계산 불필요
print(f"Model loaded on {device}")

In [ ]:
# base 모델의 특수 토큰 목록 확인
# fine-tuned 모델과 달리 <s_cord-v2> 같은 태스크 토큰이 없는 것을 직접 확인합니다.
print("등록된 특수 토큰:")
print(processor.tokenizer.all_special_tokens)

has_cord_token = "<s_cord-v2>" in processor.tokenizer.all_special_tokens
print(f"\n'<s_cord-v2>' 태스크 토큰 포함 여부: {has_cord_token}")
print("  → False = base 모델은 영수증 파싱 태스크를 모릅니다.")

## 테스트 이미지 로드

fine-tuned 노트북과 **동일한** CORD-v2 테스트 영수증을 사용해, 같은 입력에 대한 base 모델의 출력을 비교할 수 있게 합니다.

In [ ]:
# CORD-v2: 영수증 이미지 + JSON 어노테이션 데이터셋
# test[:1] → 테스트셋 첫 번째 샘플 1장만 로드
dataset = load_dataset("naver-clova-ix/cord-v2", split="test[:1]")

image = dataset[0]["image"]
ground_truth = json.loads(dataset[0]["ground_truth"])

print(f"이미지 크기: {image.size}  모드: {image.mode}")

plt.figure(figsize=(6, 8))
plt.imshow(image)
plt.axis("off")
plt.title("Test Image (CORD-v2)")
plt.show()

## 추론 (Inference)

base 모델에는 태스크 토큰이 없으므로, fine-tuned 노트북처럼 `<s_cord-v2>` 를 시작 토큰으로 쓸 수 없습니다.

대신 모델 config에 정의된 **기본 디코더 시작 토큰**(`decoder_start_token_id`, 보통 `<s>` BOS 토큰)으로 생성을 시작합니다. 이는 base 모델이 사전학습된 "텍스트 읽기" 방식 그대로 동작하게 합니다.

In [ ]:
# base 모델은 태스크 토큰이 없으므로 BOS 토큰(<s>)으로 디코딩 시작
# fine-tuned 모델과 달리 donut-base는 config에 decoder_start_token_id가
# 설정돼 있지 않으므로, 없으면 토크나이저의 bos_token_id로 대체합니다.
start_token_id = getattr(model.config, "decoder_start_token_id", None)
if start_token_id is None:
    start_token_id = processor.tokenizer.bos_token_id
decoder_input_ids = torch.full(
    (1, 1), start_token_id, dtype=torch.long, device=device
)
print("디코더 시작 토큰:", processor.tokenizer.convert_ids_to_tokens([start_token_id]))

# processor: 이미지를 모델 입력 크기로 리사이즈 후 [-1, 1] 범위로 정규화
pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

with torch.no_grad():  # 추론 시 gradient 계산 생략 → 메모리 절약
    outputs = model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_length=model.decoder.config.max_position_embeddings,  # 최대 시퀀스 길이
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,            # EOS 도달 시 생성 중단
        use_cache=True,                                            # KV-cache로 디코딩 속도 향상
        bad_words_ids=[[processor.tokenizer.unk_token_id]],       # UNK 토큰 생성 금지
        return_dict_in_generate=True,
    )

# 생성된 토큰 ID → 문자열로 디코딩
sequence = processor.batch_decode(outputs.sequences)[0]
# 특수 토큰(EOS, PAD) 제거
sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(
    processor.tokenizer.pad_token, ""
)
# 맨 앞 시작 토큰(<s> 등) 1개 제거
sequence = re.sub(r"<.*?>", "", sequence, count=1).strip()
print("\nRaw output:", repr(sequence))

### 출력 해석

fine-tuned 모델은 `<s_menu><s_nm>...</s_nm>...` 처럼 **구조화된 XML-like 토큰**을 출력합니다.

base 모델은 이런 구조를 학습한 적이 없으므로, 위 `Raw output` 은 보통 다음 중 하나입니다:
- 이미지에서 읽어낸 **단순 텍스트 나열** (사전학습 목표인 pseudo-OCR 결과)
- 의미 없는 토큰 반복 또는 빈 문자열

아래 `token2json` 으로 파싱을 시도하면, 구조가 없으므로 **빈 dict `{}` 또는 의미 없는 결과**가 나오는 것을 확인할 수 있습니다.

In [ ]:
def token2json(tokens, is_inner_value=False):
    """XML-like 토큰 시퀀스를 Python dict로 변환
    예: <s_total>12500</s_total> → {"total": "12500"}
    중첩 구조(dict 안의 dict)도 재귀적으로 처리
    """
    output = {}
    while tokens:
        start_token = re.search(r"<s_(.*?)>", tokens)
        if not start_token:
            break
        key = start_token.group(1)
        end_token = re.search(rf"</s_{key}>", tokens)
        value = tokens[start_token.end(): end_token.start() if end_token else len(tokens)]
        value = value.strip()
        if re.search(r"<s_", value):          # 중첩 태그가 있으면 재귀 파싱
            value = token2json(value, is_inner_value=True)
            if value:
                output[key] = value
        else:
            output[key] = value               # 리프 노드: 문자열 값 저장
        tokens = tokens[end_token.end():].strip() if end_token else ""
    return output

result = token2json(sequence)
print("Parsed result (base model):")
print(json.dumps(result, indent=2, ensure_ascii=False))

if not result:
    print("\n→ 빈 dict: base 모델은 구조화된 JSON 토큰을 생성하지 못했습니다 (예상된 결과).")

## 결론: 왜 Fine-tuning이 필요한가

같은 영수증 이미지를 줘도:

| | Base Model | Fine-tuned Model |
|------|------------|------------------|
| 출력 형태 | 텍스트 나열 / 무의미 토큰 | `<s_menu>...</s_menu>` 구조화 토큰 |
| `token2json` 결과 | 빈 dict 또는 의미 없음 | 정답에 가까운 dict |
| 이유 | 영수증 파싱을 **학습한 적 없음** | CORD-v2로 **fine-tuning 완료** |

즉, 사전학습된 base 모델은 "읽기" 능력만 있고, 우리가 원하는 **특정 출력 형식(JSON 스키마)** 은 fine-tuning을 통해서만 얻을 수 있습니다.

→ 커스텀 데이터로 fine-tuning 하는 전체 과정은 [`donut_training.ipynb`](./donut_training.ipynb) 참고.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

axes[0].imshow(image)
axes[0].axis("off")
axes[0].set_title("Input Image")

axes[1].axis("off")
axes[1].set_title("Base Model Output (parsing likely fails)")
axes[1].text(
    0.05, 0.95,
    json.dumps(result, indent=2, ensure_ascii=False) or "{}  (empty)",
    transform=axes[1].transAxes,
    fontsize=8,
    verticalalignment="top",
    fontfamily="monospace"
)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

axes[0].imshow(image)
axes[0].axis("off")
axes[0].set_title("Input Image")

axes[1].axis("off")
axes[1].set_title("Base Model Output (파싱 실패 예상)")
axes[1].text(
    0.05, 0.95,
    json.dumps(result, indent=2, ensure_ascii=False) or "{}  (빈 결과)",
    transform=axes[1].transAxes,
    fontsize=8,
    verticalalignment="top",
    fontfamily="monospace"
)

plt.tight_layout()
plt.show()